# Stage 3 of 3 — Optimize & Analyze

**This notebook does ONE thing**: apply the smell detector + exact-fidelity
verified repair (`rules.json` from Stage 2) to sample circuits, and report the
results. This is the cheap, seconds-not-minutes stage — the one actually worth
iterating on directly, which is the whole point of splitting it out.

**Setup — two separate inputs:**
1. **Add Input** → search `mnisq-optbench-pairs` → Add (for sample circuits to
   test against — same as Stage 1).
2. **Add Input → Notebook Output → (your username) → `02-rule-building`** (the
   saved version of Stage 2, for `rules.json`).
3. **Internet: On**.

No mining, no rule-building happens here — if either input is missing, the
cells below fail fast with a clear message telling you which one.

In [ ]:
import glob
print(glob.glob('/kaggle/input/*'))
print(glob.glob('/kaggle/input/**/*', recursive=True)[:20])

In [ ]:
# Base package only -- same as Stage 2, no [mining] extra needed here.
!pip install -q "quantum-circuit-smell-intelligence @ git+https://github.com/veerakrish/quantum-circuit-smell-intelligence.git"

In [ ]:
try:
    import qcs_pipeline
    print("qcs_pipeline imported OK from:", qcs_pipeline.__file__)
except ModuleNotFoundError as e:
    raise RuntimeError(
        "qcs_pipeline is not importable. Check the pip install cell's output "
        "above, or restart the session if you just picked up a code update."
    ) from e

In [ ]:
import glob
from pathlib import Path

rule_matches = glob.glob("/kaggle/input/**/rules.json", recursive=True)
if not rule_matches:
    raise FileNotFoundError(
        "rules.json not found under /kaggle/input. Did you attach Stage 2's "
        "saved notebook output? (Add Input -> Notebook Output -> your "
        "username -> 02-rule-building)"
    )
RULES_PATH = Path(rule_matches[0])
print(f"Using {RULES_PATH}")

from qcs_pipeline.mining.from_kaggle_pairs import find_pair_chunks
try:
    chunk_files = find_pair_chunks(Path("/kaggle/input"))
except FileNotFoundError as e:
    raise FileNotFoundError(
        "mnisq-optbench-pairs dataset not found under /kaggle/input. Did you "
        "attach it? (Add Input -> search mnisq-optbench-pairs -> Add)"
    ) from e
print(f"Using {len(chunk_files)} sample-circuit chunk files, e.g. {chunk_files[0]}")

In [ ]:
import pandas as pd

df_sample = pd.read_parquet(chunk_files[0], columns=["base_id", "input_qasm"]).head(5)
df_sample

In [ ]:
# The only real work this notebook does: run the verified optimizer over a
# handful of sample circuits and report before/after gate counts, fidelity,
# and how many rule applications were quarantined by the exact-fidelity check.
from qcs_pipeline.pipeline import QuantumCircuitSmellOptimizer

optimizer = QuantumCircuitSmellOptimizer(rule_db_path=RULES_PATH)

for row in df_sample.itertuples(index=False):
    result = optimizer.optimize(row.input_qasm)
    print(
        f"{row.base_id}: {result.n_gates_before} -> {result.n_gates_after} gates "
        f"(fidelity={result.fidelity:.10f}, applications={len(result.applied_smells)}, "
        f"quarantined={result.quarantined_rule_count})"
    )

## Interpreting results

- **`fidelity` should read `1.0000000000` (or extremely close, floating-point
  noise).** That's the exact-fidelity verification working — any rule
  application that would have broken it gets bisected out and quarantined
  instead, never silently applied.
- **`applications=0` is expected for many circuits** given Stage 2's
  breakdown (most mined rules are `structural_only`, not `applicable`) — a
  small `applicable` rule count means many circuits simply won't match any
  of them. That's a real ceiling on this dataset's rule set, not a bug here.
- **`quarantined > 0`** means a rule that looked safe (passed Stage 2's
  filters) still failed the exact check on this specific circuit — worth
  noting which `base_id` triggered it if you want to dig into why later.